<a href="https://colab.research.google.com/github/ralph-Jung/AI-Class/blob/main/4%EC%A3%BC%EC%B0%A8/10_5_diabete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


# 1. 데이터 불러오기
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/diabetes.csv")

print(df.head())
print(df.shape)


# 2. 입력 / 정답 분리
# 이 문제는 Outcome 을 통해서 이진 분류 하는 거임
# 그렇기에 마지막 출력층에 활성화 함수로 0과 1 사이의 값이 나오도록 하는 시그모이드 함수를 사용한다
X = df.drop("Outcome", axis=1)
y = df["Outcome"].values


# 3. 학습 / 테스트 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# 4. 결측치 처리 (0으로 기록된 값 -> 학습 데이터의 중앙값으로 채우기)
# 혈당, 혈압, 피부 두께, 인슐린, BMI는 0일 수 없으므로 결측으로 본다.
cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

X_train = X_train.copy()
X_test = X_test.copy()
X_train[cols] = X_train[cols].replace(0, np.nan)
X_test[cols] = X_test[cols].replace(0, np.nan)

medians = X_train[cols].median()          # 학습 데이터 기준 중앙값
X_train[cols] = X_train[cols].fillna(medians)
X_test[cols] = X_test[cols].fillna(medians)


# 5. 데이터 정규화
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# 6. 모델 생성 (입력 8 -> 은닉층 32 -> 은닉층 16 -> 출력 1)
# 첫번째 레이어 : 뉴런 32 개 , 활성화 함수 : relu
# 두번째 레이어 : 뉴런 16개 , 활성화 함수 : relu
# 세번째 레이어 : 뉴런 1개 , 활성화 함수 : sigmoid
# 출력층의 활성화 함수는 해당 문제가 이진 분류 , 다중 분류 , 회귀냐 에 따라서 달라진다
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(8,)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])


# 7. 모델 설정 (한 번만)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])


# 8. 모델 구조 출력
model.summary()


# 9. 모델 학습 (검증 손실이 10 에포크 동안 좋아지지 않으면 멈추고 최적 가중치로 복원)
# EarlyStopping은 val_loss 하나만 지켜보고 한 애폭 동안 가장 작은 손실을 내면 그 가중치를 기억해 두고 그 이후로 10 애폭 동안 손실이 최적일 때 보다 낮아지지 않으면 중단하고 해당 가중치로 되돌리고 학습을 중단함

early = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)

# 에포크마다의 손실과 정확도를 기록해 둔 것
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early],
    verbose=1,
)


# 10. 테스트 데이터 예측
# 9번에서 나온 최적의 가중치를 가진 모델을 바탕으로 test 데이터를 넣어서 예측을 진행한다
# 0.5 기준은 sigmoid(이진 분류, 다중 레이블)에서 쓰는 방식
# softmax는 세 확률 중 가장 큰 것의 위치를 골라야하므로 argmax 를 이용
y_prob = model.predict(X_test)
y_pred = (y_prob >= 0.5).astype(int).flatten()


# 11. 성능 평가
# y_test 에는 실제 정답값 , y_pred 에는 예측한 결과를 넣고 이런식으로 전체 중에 몇 개를 제대로 예측했는지 계산하는 것이다
# y_test = [1, 0, 1, 1, 0]   # 실제 정답
# y_pred = [1, 0, 0, 1, 1]   # 모델 예측 (0.5 기준으로 자른 값)
accuracy = accuracy_score(y_test, y_pred)
print("\nAccuracy =", accuracy)
print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report")
print(classification_report(y_test, y_pred))

   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  
(768, 9)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 833 (3.25 KB)

 Trainable params: 833 (3.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.6680 - loss: 0.6540 - val_accuracy: 0.7073 - val_loss: 0.6429
Epoch 2/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7067 - loss: 0.5953 - val_accuracy: 0.7480 - val_loss: 0.5836
Epoch 3/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7251 - loss: 0.5564 - val_accuracy: 0.7724 - val_loss: 0.5473
Epoch 4/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7454 - loss: 0.5281 - val_accuracy: 0.7886 - val_loss: 0.5236
Epoch 5/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7454 - loss: 0.5078 - val_accuracy: 0.7805 - val_loss: 0.5057
Epoch 6/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7719 - loss: 0.4920 - val_accuracy: 0.7805 - val_loss: 0.4905
Epoch 7/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7800 - loss: 0.4796 - val_accuracy: 0.7967 - val_loss: 0.4776
Epoch 8/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7902 - loss: 0.4689 - val_accuracy: 0.7886 - 

1/5 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step

Accuracy = 0.7467532467532467

Confusion Matrix
[[82 18]
 [21 33]]

Classification Report
              precision    recall  f1-score   support

           0       0.80      0.82      0.81       100
           1       0.65      0.61      0.63        54

    accuracy                           0.75       154
   macro avg       0.72      0.72      0.72       154
weighted avg       0.74      0.75      0.75       154

